## LangChain이란 LLM을 여러 기능과 연결해서 하나의 애플리케이션으로 만들기 쉽게 해주는 프레임 워크


In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_core.output_parsers import PydanticOutputParser


class CommentModeration(BaseModel):
    toxicity: Literal["safe", "warning", "toxic"] = Field(
        description="댓글의 악성 정도. safe는 안전, warning은 주의가 필요한 댓글, toxic은 악성 댓글"
    )

    contains_profanity: bool = Field(
        description="욕설이나 비속어가 포함되어 있는지 여부"
    )

    contains_personal_attack: bool = Field(
        description="특정 개인이나 대상을 직접적으로 공격하거나 비하하는지 여부"
    )

    reason: Optional[str] = Field(
        default=None, description="악성 요소가 있다면 그 이유"
    )


llm = ChatOllama(model="gemma3:4b", temperature=0.0)

parses = PydanticOutputParser(pydantic_object=CommentModeration)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 SNS에 댓글을 분석하여 악성 댓글을 자동으로 필터링 해주는 AI야. \n\n{format_instructions}",
        ),
        ("human", "댓글 : {comment}"),
    ]
).partial(format_instructions=parses.get_format_instructions())

chain = prompt | llm | parses


comments = [
    "제품이 정말 마음에 들어요. 배송도 빠르고 포장도 깔끔했습니다.",
    "이딴 것도 상품이라고 파냐? 진짜 개같네.",
    "판매자님 설명도 제대로 안 읽고 물건 보내시나요? 일을 이렇게밖에 못하세요?",
    "제품은 괜찮은데 가격이 조금 비싼 것 같아요.",
    "서비스가 너무 엉망이네요. 담당자는 진짜 무능한 것 같습니다.",
    "와 진짜 존나 맛없어요. 돈 아까워 죽겠네.",
    "상품 자체는 괜찮지만 직원 응대가 너무 불친절했습니다.",
    "판매자 너는 장사할 자격이 없다. 머리가 있으면 이런 식으로 운영하지 마라.",
]

for comment in comments:
    res = chain.invoke({"comment": comment})
    print(f"'{comment}")
    print(f"toxicity : {res.toxicity}")
    print(f"욕설 포함 여부 : {res.contains_profanity}")
    print(f"인신 공격 여부 : {res.contains_personal_attack}")
    if res.reason:
        print(f"문제가 있을 때 그 이유 : {res.reason}")
    print()

'제품이 정말 마음에 들어요. 배송도 빠르고 포장도 깔끔했습니다.
toxicity : safe
욕설 포함 여부 : False
인신 공격 여부 : False

'이딴 것도 상품이라고 파냐? 진짜 개같네.
toxicity : toxic
욕설 포함 여부 : True
인신 공격 여부 : False
문제가 있을 때 그 이유 : 비속어 사용 및 부정적인 감정 표현

'판매자님 설명도 제대로 안 읽고 물건 보내시나요? 일을 이렇게밖에 못하세요?
toxicity : toxic
욕설 포함 여부 : False
인신 공격 여부 : True
문제가 있을 때 그 이유 : 특정 판매자를 비하하는 표현과 함께 직접적인 공격적인 발언을 포함하고 있어 악성 댓글로 판단됨.

'제품은 괜찮은데 가격이 조금 비싼 것 같아요.
toxicity : safe
욕설 포함 여부 : False
인신 공격 여부 : False

'서비스가 너무 엉망이네요. 담당자는 진짜 무능한 것 같습니다.
toxicity : warning
욕설 포함 여부 : False
인신 공격 여부 : True
문제가 있을 때 그 이유 : 담당자에 대한 비판적인 언급 및 무능함에 대한 표현으로, 개인적인 비난의 뉘앙스를 포함하고 있어 주의가 필요합니다.

'와 진짜 존나 맛없어요. 돈 아까워 죽겠네.
toxicity : toxic
욕설 포함 여부 : True
인신 공격 여부 : False
문제가 있을 때 그 이유 : 비속어 사용 및 부정적인 평가로 인해 악성 댓글로 판단됨

'상품 자체는 괜찮지만 직원 응대가 너무 불친절했습니다.
toxicity : safe
욕설 포함 여부 : False
인신 공격 여부 : False

'판매자 너는 장사할 자격이 없다. 머리가 있으면 이런 식으로 운영하지 마라.
toxicity : toxic
욕설 포함 여부 : False
인신 공격 여부 : True
문제가 있을 때 그 이유 : 판매자를 직접적으로 비난하고 운영 방식에 대한 부정적인 평가를 내렸다. 공격적인 표현을 사용했다.



In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_core.output_parsers import PydanticOutputParser


class CommentModeration(BaseModel):
    toxicity: Literal["safe", "warning", "toxic"] = Field(
        description="댓글의 악성 정도. safe는 안전, warning은 주의가 필요한 댓글, toxic은 악성 댓글"
    )

    contains_profanity: bool = Field(
        description="욕설이나 비속어가 포함되어 있는지 여부"
    )

    contains_personal_attack: bool = Field(
        description="특정 개인이나 대상을 직접적으로 공격하거나 비하하는지 여부"
    )

    reason: Optional[str] = Field(
        default=None, description="악성 요소가 있다면 그 이유"
    )


llm = ChatOllama(model="gemma3:4b", temperature=0.0)

parses = PydanticOutputParser(pydantic_object=CommentModeration)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "너는 SNS에 댓글을 분석하여 악성 댓글을 자동으로 필터링 해주는 AI야. \n\n{format_instructions}",
        ),
        ("human", "댓글 : {comment}"),
    ]
).partial(format_instructions=parses.get_format_instructions())

chain = prompt | llm | parses


comments = [
    "제품이 정말 마음에 들어요. 배송도 빠르고 포장도 깔끔했습니다.",
    "이딴 것도 상품이라고 파냐? 진짜 개같네.",
    "판매자님 설명도 제대로 안 읽고 물건 보내시나요? 일을 이렇게밖에 못하세요?",
    "제품은 괜찮은데 가격이 조금 비싼 것 같아요.",
    "서비스가 너무 엉망이네요. 담당자는 진짜 무능한 것 같습니다.",
    "와 진짜 존나 맛없어요. 돈 아까워 죽겠네.",
    "상품 자체는 괜찮지만 직원 응대가 너무 불친절했습니다.",
    "판매자 너는 장사할 자격이 없다. 머리가 있으면 이런 식으로 운영하지 마라.",
]

for comment in comments:
    res = chain.invoke({"comment": comment})
    print(f"'{comment}")
    print(f"toxicity : {res.toxicity}")
    print(f"욕설 포함 여부 : {res.contains_profanity}")
    print(f"인신 공격 여부 : {res.contains_personal_attack}")
    if res.reason:
        print(f"문제가 있을 때 그 이유 : {res.reason}")
    print()

'제품이 정말 마음에 들어요. 배송도 빠르고 포장도 깔끔했습니다.
toxicity : safe
욕설 포함 여부 : False
인신 공격 여부 : False

'이딴 것도 상품이라고 파냐? 진짜 개같네.
toxicity : toxic
욕설 포함 여부 : True
인신 공격 여부 : False
문제가 있을 때 그 이유 : 비속어 사용 및 부정적인 감정 표현

'판매자님 설명도 제대로 안 읽고 물건 보내시나요? 일을 이렇게밖에 못하세요?
toxicity : toxic
욕설 포함 여부 : False
인신 공격 여부 : True
문제가 있을 때 그 이유 : 특정 판매자를 비하하는 표현과 함께 직접적인 공격적인 발언을 포함하고 있어 악성 댓글로 판단됨.

'제품은 괜찮은데 가격이 조금 비싼 것 같아요.
toxicity : safe
욕설 포함 여부 : False
인신 공격 여부 : False

'서비스가 너무 엉망이네요. 담당자는 진짜 무능한 것 같습니다.
toxicity : warning
욕설 포함 여부 : False
인신 공격 여부 : True
문제가 있을 때 그 이유 : 담당자에 대한 비판적인 언급 및 무능함에 대한 표현으로, 개인적인 비난의 뉘앙스를 포함하고 있어 주의가 필요합니다.

'와 진짜 존나 맛없어요. 돈 아까워 죽겠네.
toxicity : toxic
욕설 포함 여부 : True
인신 공격 여부 : False
문제가 있을 때 그 이유 : 비속어 사용 및 부정적인 평가로 인해 악성 댓글로 판단됨

'상품 자체는 괜찮지만 직원 응대가 너무 불친절했습니다.
toxicity : safe
욕설 포함 여부 : False
인신 공격 여부 : False

'판매자 너는 장사할 자격이 없다. 머리가 있으면 이런 식으로 운영하지 마라.
toxicity : toxic
욕설 포함 여부 : False
인신 공격 여부 : True
문제가 있을 때 그 이유 : 판매자를 직접적으로 비난하고 운영 방식에 대한 부정적인 평가를 내렸다. 공격적인 표현을 사용했다.



In [4]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from typing import Literal, Optional
from langchain_core.output_parsers import PydanticOutputParser


# 스키마 — 카테고리 + 개선 제안 추가
class ReviewFull(BaseModel):
    rating: int = Field(description="평점 1-5점", ge=1, le=5)
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="감정")
    category: Literal["품질", "가격", "배송", "디자인", "기타"] = Field(
        description="리뷰가 다루는 영역"
    )
    keywords: list[str] = Field(description="핵심 키워드 3개")
    improvement: Optional[str] = Field(
        default=None, description="개선 제안 (있으면, 없으면 None)"
    )


llm = ChatOllama(model="gemma3:4b", temperature=0.7)


parser_r = PydanticOutputParser(pydantic_object=ReviewFull)
prompt_r = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "쇼핑몰 리뷰 분석가입니다. 한국어로 답하세요.\n\n{format_instructions}",
        ),
        ("human", "리뷰: {review}"),
    ]
).partial(format_instructions=parser_r.get_format_instructions())

chain_review = prompt_r | llm | parser_r


# 테스트 — 3개 리뷰
reviews = [
    "포장이 너무 아쉽네요. 박스가 찌그러져 왔어요.",
    "가성비 짱! 이 가격에 이 정도면 최고예요.",
    "디자인은 예쁜데 기능이 좀 부족하네요.",
]

for r in reviews:
    res = chain_review.invoke({"review": r})
    print(f"📝 '{r}'")
    print(f"   ⭐ {res.rating}/5 | {res.sentiment} | 📁 {res.category}")
    print(f"   🏷️  {', '.join(res.keywords)}")
    if res.improvement:
        print(f"   💡 개선: {res.improvement}")
    print()

📝 '포장이 너무 아쉽네요. 박스가 찌그러져 왔어요.'
   ⭐ 2/5 | negative | 📁 배송
   🏷️  포장, 박스, 찌그러짐

📝 '가성비 짱! 이 가격에 이 정도면 최고예요.'
   ⭐ 5/5 | positive | 📁 가격
   🏷️  가성비, 가격, 최고

📝 '디자인은 예쁜데 기능이 좀 부족하네요.'
   ⭐ 3/5 | negative | 📁 디자인
   🏷️  디자인, 기능, 예쁜



In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory

store = {}


llm = ChatOllama(model="gemma3:4b", temperature=0.7)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 {persona}입니다."),
        MessagesPlaceholder(variable_name="history"),
        ("human", "{question}"),
    ]
)


def get_session_id_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chain = RunnableWithMessageHistory(
    prompt | llm,
    get_session_id_history,
    input_messages_key="question",
    history_messages_key="history",
)


result = chain.invoke(
    {
        "question": "AI 엔지니어 직무를 꿈꾸는 신입들이 뭘 준비하면 좋을까 핵심역량으로",
        "persona": "AI 엔지니어를 10년이상 직무를 수행한 전문가",
    },
    config={"configurable": {"session_id": "test_01"}},
)

print(result.content)

오랜 기간 AI 엔지니어링 분야에서 일해 온 경험을 바탕으로, AI 엔지니어 직무를 꿈꾸는 신입들이 핵심 역량을 키우기 위해 무엇을 준비해야 하는지 핵심적으로 정리해 드리겠습니다. 

**1. 탄탄한 기초 다지기:**

* **수학 & 통계:** AI의 핵심은 수학과 통계입니다. 선형대수, 미적분, 확률 및 통계학에 대한 깊이 있는 이해는 필수입니다.
    * **추천 학습:** MIT OpenCourseware의 Linear Algebra, Calculus, Probability & Statistics 강의를 통해 기초를 다지고, 다양한 문제 풀이를 통해 실력을 향상시키세요.
* **프로그래밍:** Python은 AI 개발에 가장 널리 사용되는 언어입니다. 능숙하게 다룰 수 있도록 꾸준히 연습해야 합니다.
    * **추천 학습:** Codecademy, Coursera, Udemy 등의 온라인 강의를 통해 Python 기초부터 시작하고, 프로젝트를 통해 실전 경험을 쌓으세요.
* **자료구조 & 알고리즘:** 효율적인 AI 모델 개발을 위해 자료구조와 알고리즘에 대한 이해는 매우 중요합니다.
    * **추천 학습:** LeetCode, HackerRank 등의 플랫폼을 통해 알고리즘 문제 풀이를 꾸준히 하세요.

**2. AI 핵심 기술 습득:**

* **머신러닝:** 지도 학습, 비지도 학습, 강화 학습 등 기본적인 머신러닝 개념과 알고리즘을 이해하고, scikit-learn, TensorFlow, PyTorch 등 주요 머신러닝 라이브러리를 활용하는 방법을 익히세요.
    * **추천 학습:** Andrew Ng 교수의 Machine Learning Coursera 강의는 머신러닝 입문자에게 매우 유익합니다.
* **딥러닝:** 인공 신경망, CNN, RNN 등 딥러닝 모델의 구조와 작동 원리를 이해하고, 딥러닝 프레임워크를 활용하여 실제 모델을 구축해 보세요.
* **자연어 처리 (NLP):** 텍스트 데이터를 분석하고 이해하는 기술입니다.